In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import copy

In [15]:
train = pd.read_csv('train_fe.csv')
test = pd.read_csv('test_fe.csv')

# feature engineering, the same as in advanced catboost
compound_max_life = train.groupby('Compound')['TyreLife'].max().reset_index()
compound_max_life.rename(columns={'TyreLife': 'Track_Compound_Max_Life'}, inplace=True)

train['is_train'] = 1
test['is_train'] = 0
df = pd.concat([train, test], ignore_index=True)

df = df.merge(compound_max_life, on='Compound', how='left')

# Zapamiętanie oryginalnej kolejności dla zbioru testowego!
df['orig_index'] = np.arange(len(df))
df = df.sort_values(by=['Race', 'Year', 'Driver', 'LapNumber'])

# Cechy inżynieryjne
df['Grid_Avg_TyreLife'] = df.groupby(['Race', 'Year', 'LapNumber'])['TyreLife'].transform('mean')
df['TyreLife_vs_Grid'] = df['TyreLife'] - df['Grid_Avg_TyreLife']
df['Normalized_TyreLife_Squared'] = (df['TyreLife'] / (df['Track_Compound_Max_Life'] + 1e-5)) ** 2
df['Panic_Indicator'] = df['LapTime_Delta'] * df['Position_Change']
df['Panic_Indicator'] = df['Panic_Indicator'].apply(lambda x: x if x > 0 else 0)
df['Gap_To_Fastest_Rolling3'] = df.groupby(['Race', 'Year', 'Driver'])['Gap_To_Fastest'].transform(lambda x: x.rolling(3, min_periods=1).mean())
df['LapTime_Delta_Rolling3_Sum'] = df.groupby(['Race', 'Year', 'Driver'])['LapTime_Delta'].transform(lambda x: x.rolling(3, min_periods=1).sum())

df.drop(columns=['Grid_Avg_TyreLife', 'Track_Compound_Max_Life'], inplace=True, errors='ignore')

df = df.sort_values(by='orig_index').drop(columns=['orig_index'])

train_df = df[df['is_train'] == 1].drop(columns=['is_train'])
test_df = df[df['is_train'] == 0].drop(columns=['is_train', 'PitNextLap'])


In [19]:
X = train_df.drop(columns=['PitNextLap'])
y = train_df['PitNextLap'].values
X_test = test_df.copy()


In [20]:
# preparing data
X.drop(columns=['id'], inplace=True, errors='ignore')
X_test.drop(columns=['id'], inplace=True, errors='ignore')

cat_cols = ['Driver', 'Compound', 'Race', 'Year']
cont_cols = [c for c in X.columns if c not in cat_cols]


X[cont_cols] = X[cont_cols].fillna(0)
X_test[cont_cols] = X_test[cont_cols].fillna(0)

# StandardScaler
scaler = StandardScaler()
X[cont_cols] = scaler.fit_transform(X[cont_cols])
X_test[cont_cols] = scaler.transform(X_test[cont_cols])

# cat cols
cat_dims = []
for col in cat_cols:
    X[col] = X[col].astype(str).fillna('missing')
    X_test[col] = X_test[col].astype(str).fillna('missing')

    le = LabelEncoder()
    le.fit(pd.concat([X[col], X_test[col]]))

    X[col] = le.transform(X[col])
    X_test[col] = le.transform(X_test[col])

    num_classes = len(le.classes_)
    embed_dim = max(2, min(50, num_classes // 2))

    cat_dims.append((num_classes, embed_dim))

print("Rozmiary zmiennych kategorycznych (vocab_size, embed_dim):", cat_dims)
print("Liczba zmiennych ciągłych:", len(cont_cols))

Rozmiary zmiennych kategorycznych (vocab_size, embed_dim): [(887, 50), (5, 2), (26, 13), (4, 2)]
Liczba zmiennych ciągłych: 19


In [21]:
# Dataset
class F1Dataset(Dataset):
    def __init__(self, X_cat, X_cont, y=None):
        self.X_cat = torch.tensor(X_cat.values, dtype=torch.long)
        self.X_cont = torch.tensor(X_cont.values, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32) if y is not None else None

    def __len__(self):
        return len(self.X_cat)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X_cat[idx], self.X_cont[idx], self.y[idx]
        return self.X_cat[idx], self.X_cont[idx]

In [22]:
class ResidualBlock(nn.Module):
    def __init__(self, hidden_size, dropout=0.2):
        super(ResidualBlock, self).__init__()
        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.bn1 = nn.BatchNorm1d(hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.bn2 = nn.BatchNorm1d(hidden_size)

    def forward(self, x):
        residual = x
        out = self.fc1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        out = self.bn2(out)
        out += residual
        out = self.relu(out)
        return out

class TabularResNet(nn.Module):
    def __init__(self, cat_dims, num_cont, hidden_size=256, num_blocks=3, dropout=0.2):
        super(TabularResNet, self).__init__()

        # Embeddings
        self.embeddings = nn.ModuleList([nn.Embedding(vocab, dim) for vocab, dim in cat_dims])
        total_embed_dim = sum([dim for _, dim in cat_dims])

        # Init layer
        self.initial_layer = nn.Sequential(
            nn.Linear(total_embed_dim + num_cont, hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # res blocks
        self.res_blocks = nn.Sequential(*[ResidualBlock(hidden_size, dropout) for _ in range(num_blocks)])

        # output
        self.output_layer = nn.Linear(hidden_size, 1)

    def forward(self, x_cat, x_cont):
        embed_outs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.cat(embed_outs + [x_cont], dim=1)

        x = self.initial_layer(x)
        x = self.res_blocks(x)
        x = self.output_layer(x)
        return x.squeeze(1)

In [23]:
EPOCHS = 15
BATCH_SIZE = 1024
LR = 0.001
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds_nn = np.zeros(len(X))
test_preds_nn = np.zeros(len(X_test))

test_dataset = F1Dataset(X_test[cat_cols], X_test[cont_cols])
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- FOLD {fold+1} ---")

    X_tr_cat, X_tr_cont, y_tr = X[cat_cols].iloc[train_idx], X[cont_cols].iloc[train_idx], y[train_idx]
    X_va_cat, X_va_cont, y_va = X[cat_cols].iloc[val_idx], X[cont_cols].iloc[val_idx], y[val_idx]

    train_dataset = F1Dataset(X_tr_cat, X_tr_cont, y_tr)
    val_dataset = F1Dataset(X_va_cat, X_va_cont, y_va)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)


    model = TabularResNet(cat_dims, len(cont_cols)).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

    num_pos = y_tr.sum()
    num_neg = len(y_tr) - num_pos
    pos_weight = torch.tensor([num_neg / num_pos], dtype=torch.float32).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_auc = 0
    best_model_weights = None

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0

        for batch_cat, batch_cont, batch_y in train_loader:
            batch_cat, batch_cont, batch_y = batch_cat.to(DEVICE), batch_cont.to(DEVICE), batch_y.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(batch_cat, batch_cont)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # validation
        model.eval()
        val_preds = []
        with torch.no_grad():
            for batch_cat, batch_cont, _ in val_loader:
                batch_cat, batch_cont = batch_cat.to(DEVICE), batch_cont.to(DEVICE)
                outputs = model(batch_cat, batch_cont)
                probs = torch.sigmoid(outputs) # 0-1
                val_preds.extend(probs.cpu().numpy())

        val_auc = roc_auc_score(y_va, val_preds)
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss/len(train_loader):.4f} | Val AUC: {val_auc:.5f}")

        if val_auc > best_auc:
            best_auc = val_auc
            best_model_weights = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_model_weights)
    model.eval()

    # oof val
    fold_val_preds = []
    with torch.no_grad():
        for batch_cat, batch_cont, _ in val_loader:
            batch_cat, batch_cont = batch_cat.to(DEVICE), batch_cont.to(DEVICE)
            outputs = model(batch_cat, batch_cont)
            fold_val_preds.extend(torch.sigmoid(outputs).cpu().numpy())
    oof_preds_nn[val_idx] = fold_val_preds

    # test
    fold_test_preds = []
    with torch.no_grad():
        for batch_cat, batch_cont in test_loader:
            batch_cat, batch_cont = batch_cat.to(DEVICE), batch_cont.to(DEVICE)
            outputs = model(batch_cat, batch_cont)
            fold_test_preds.extend(torch.sigmoid(outputs).cpu().numpy())

    test_preds_nn += np.array(fold_test_preds) / skf.n_splits
    print(f"Best AUC Foldu {fold+1}: {best_auc:.5f}")

print(f"\n=== FINAL OOF ROC AUC (Tabular ResNet): {roc_auc_score(y, oof_preds_nn):.5f} ===")


Device: cuda

--- FOLD 1 ---
Epoch 1/15 | Train Loss: 0.6992 | Val AUC: 0.92164
Epoch 2/15 | Train Loss: 0.5707 | Val AUC: 0.92780
Epoch 3/15 | Train Loss: 0.5499 | Val AUC: 0.93042
Epoch 4/15 | Train Loss: 0.5349 | Val AUC: 0.93322
Epoch 5/15 | Train Loss: 0.5272 | Val AUC: 0.93482
Epoch 6/15 | Train Loss: 0.5186 | Val AUC: 0.93565
Epoch 7/15 | Train Loss: 0.5094 | Val AUC: 0.93713
Epoch 8/15 | Train Loss: 0.5025 | Val AUC: 0.93681
Epoch 9/15 | Train Loss: 0.4972 | Val AUC: 0.93812
Epoch 10/15 | Train Loss: 0.4915 | Val AUC: 0.93799
Epoch 11/15 | Train Loss: 0.4875 | Val AUC: 0.93856
Epoch 12/15 | Train Loss: 0.4832 | Val AUC: 0.93891
Epoch 13/15 | Train Loss: 0.4786 | Val AUC: 0.93920
Epoch 14/15 | Train Loss: 0.4770 | Val AUC: 0.93986
Epoch 15/15 | Train Loss: 0.4706 | Val AUC: 0.93950
Best AUC Foldu 1: 0.93986

--- FOLD 2 ---
Epoch 1/15 | Train Loss: 0.7275 | Val AUC: 0.91819
Epoch 2/15 | Train Loss: 0.5719 | Val AUC: 0.92564
Epoch 3/15 | Train Loss: 0.5474 | Val AUC: 0.92804
Epoch

In [25]:

submission = pd.read_csv('sample_submission.csv')
submission['PitNextLap'] = test_preds_nn
submission.to_csv('resnet_f1_submission.csv', index=False)

In [26]:
# DL + CATBOOST
sub_cb = pd.read_csv('catboost_optuna_submission.csv')
sub_nn = pd.read_csv('resnet_f1_submission.csv')

rank_cb = sub_cb['PitNextLap'].rank() / len(sub_cb)
rank_nn = sub_nn['PitNextLap'].rank() / len(sub_nn)

sub_final = sub_cb.copy()
sub_final['PitNextLap'] = (rank_cb * 0.65) + (rank_nn * 0.35)

sub_final.to_csv('catboost_resnet_blend_submission.csv', index=False)
